[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/student/Lab3_Memory_and_Tool_Calling_Student.ipynb)

# Lab 3: Conversation Memory & Tool Calling
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> You are building a clinical assistant for KHCC nurses. The assistant must remember the conversation context (so nurses can ask follow-up questions) and have access to tools that look up patient vitals and check reference ranges.

### Objective
- Build conversation memory using a messages list
- Implement OpenAI tool calling with clinical tools
- Create a multi-turn clinical assistant

---
### Setup

In [4]:
!pip install -q openai tiktoken

In [1]:
import json
from openai import OpenAI
from google.colab import userdata

# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)
print("Client ready!")

Client ready!


---
## Cell 1 — The Stateless Problem
First, let's see why memory matters. Make two API calls without memory.

In [6]:
# === CELL 1: THE STATELESS PROBLEM ===

# Call 1: Ask about a patient
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"}]
)
print("Response 1:", response1.choices[0].message.content)

# Call 2: Follow-up WITHOUT memory
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What should the nurse do about it?"}]
)
print("\nResponse 2:", response2.choices[0].message.content)

# TODO: Notice — the model has NO idea what "it" refers to!

Response 1: A temperature of 101.5°F (approximately 38.6°C) is considered a fever. In adults, a fever can sometimes indicate an underlying infection or other medical condition that may require further evaluation. Whether this temperature is concerning depends on several factors, including:

- The patient's overall health status and medical history
- Any accompanying symptoms (e.g., chills, cough, difficulty breathing, rash, etc.)
- The duration of the fever
- Any recent travel or exposure to contagious diseases
- The presence of any chronic conditions

In general, if the fever persists, is accompanied by severe symptoms, or if the patient has underlying health conditions, it is advisable to seek medical evaluation. For a definitive assessment, consulting a healthcare professional is recommended.

Response 2: To provide a specific and accurate response, I would need more context regarding the situation or issue that the nurse is facing. For example, are you asking about a medical concer

---
## Cell 2 — Building Conversation Memory
Maintain a messages list and send the full history each time.

In [12]:
# === CELL 2: CONVERSATION MEMORY ===

# TODO: Create a messages list with a system message
# System: "You are a clinical assistant at KHCC. Help nurses with patient care questions."
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions."}
]

# TODO: Write a function send_message(user_input) that:
# 1. Appends the user message to the messages list
# 2. Calls client.chat.completions.create() with the full messages list
# 3. Appends the assistant response to the messages list
# 4. Returns the response text

def send_message(user_input):
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    assistant_response = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_response})
    return messages

# TODO: Test with a multi-turn conversation:
# Turn 1: "Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"
# Turn 2: "What should the nurse do about it?"
# Turn 3: "Should we also check their blood pressure?"
# Notice: Turn 2 and 3 now correctly reference previous context!
print("Turn 1:", send_message("Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"))
print("\nTurn 2:", send_message("What should the nurse do about it?"))
print("\nTurn 3:", send_message("Should we also check their blood pressure?"))

# print(messages) # Uncomment to see the full message history

Turn 1: [{'role': 'system', 'content': 'You are a clinical assistant at KHCC. Help nurses with patient care questions.'}, {'role': 'user', 'content': 'Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?'}, {'role': 'assistant', 'content': "A temperature of 101.5°F is considered a mild fever. While it may not be immediately alarming, it can be concerning depending on the patient's overall condition, history, and any accompanying symptoms. Factors to consider include:\n\n- The patient's age and baseline temperature\n- Recent medical history (e.g., recent infections, surgeries)\n- Presence of other symptoms (e.g., chills, cough, pain, dizziness)\n- Any underlying health conditions that might impact their ability to fight infections\n\nIt's essential to monitor the patient's fever and assess for any changes in condition. If the fever persists, worsens, or is accompanied by significant symptoms, it may warrant further evaluation and intervention by a nurse or physician. You 

---
## Cell 3 — Token Budget Management
As conversations grow, the token count increases. Let's track it.

In [9]:
# === CELL 3: TOKEN BUDGET ===
import tiktoken

# TODO: Write a function count_conversation_tokens(messages) that
# counts the total tokens in the conversation history
# Hint: use tiktoken.encoding_for_model("gpt-4o-mini")
# Then concatenate all message contents and encode

def count_conversation_tokens(messages):
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    total_tokens = 0
    for message in messages:
        # Account for role and content. The structure adds about 4 tokens per message.
        total_tokens += len(encoding.encode(message["content"])) + 4
    return total_tokens

# TODO: After each send_message call, print the token count
# Discuss: what happens when the conversation reaches 128K tokens?

# Let's re-run Cell 2's send_message calls but with token counting.
# First, reset messages for a clean start.
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions."}
]

def send_message_with_token_count(user_input):
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    assistant_response = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_response})
    token_count = count_conversation_tokens(messages)
    print(f"Current conversation token count: {token_count}")
    return assistant_response

print("Turn 1:", send_message_with_token_count("Patient MRN 18887 has a temperature of 101.5°F. Is this concerning?"))
print("\nTurn 2:", send_message_with_token_count("What should the nurse do about it?"))
print("\nTurn 3:", send_message_with_token_count("Should we also check their blood pressure?"))


Current conversation token count: 202
Turn 1: A temperature of 101.5°F is considered a low-grade fever. In the context of patient care, the significance of a fever can depend on several factors, such as the patient's medical history, current condition, and any underlying diseases or treatments (e.g., chemotherapy, immunosuppression).

Generally, a fever may warrant further assessment, especially if:
- The patient is immunocompromised.
- There are accompanying symptoms (e.g., chills, rash, difficulty breathing).
- The patient has a history of recent infection or surgery.

It would be advisable to monitor the patient's vital signs, assess for other symptoms, and possibly notify the attending physician or healthcare provider, especially if the fever persists or escalates. Always consult with the nursing staff for immediate evaluation.
Current conversation token count: 568

Turn 2: The nurse should take the following steps in response to Patient MRN 18887's temperature of 101.5°F:

1. **As

---
## Cell 4 — Clinical Tool Definitions
Define tools the model can call to look up patient data.

In [13]:
# === CELL 4: CLINICAL TOOLS ===

# Mock patient database (simulating KHCC data)
patient_db = {
    "18887304731609": {
        "vitals": {"temperature": 101.5, "pulse": 95, "bp": "130/85", "spo2": 96, "respiration": 20},
        "ward": "KHCC-4C-HUSSAM AL-HARIRI",
        "last_updated": "2026-01-16T12:32:00"
    },
    "21307935080461": {
        "vitals": {"temperature": 98.6, "pulse": 72, "bp": "120/80", "spo2": 98, "respiration": 16},
        "ward": "KHCC-3B-ONCOLOGY",
        "last_updated": "2026-01-16T14:00:00"
    }
}

# Clinical reference ranges
reference_ranges = {
    "temperature": {"low": 97.8, "high": 99.1, "unit": "°F"},
    "pulse": {"low": 60, "high": 100, "unit": "bpm"},
    "spo2": {"low": 95, "high": 100, "unit": "%"},
    "respiration": {"low": 12, "high": 20, "unit": "breaths/min"}
}

# TODO: Define three Python functions:
# 1. lookup_vitals(mrn: str) -> dict — returns patient vitals from patient_db
#    If MRN not found, return {"error": "Patient not found"}
# 2. check_abnormal_range(vital_type: str, value: float) -> dict
#    Returns {"vital": ..., "value": ..., "status": "normal"/"high"/"low", "reference": ...}
# 3. get_patient_summary(mrn: str) -> dict — returns full patient info

def lookup_vitals(mrn: str) -> dict:
    """Looks up the latest vitals for a patient by MRN."""
    if mrn in patient_db:
        return patient_db[mrn]["vitals"]
    return {"error": "Patient not found"}

def check_abnormal_range(vital_type: str, value: float) -> dict:
    """Checks if a vital sign value is within normal clinical range."""
    if vital_type not in reference_ranges:
        return {"error": "Vital type not recognized"}

    ref = reference_ranges[vital_type]
    status = "normal"
    if value < ref["low"]:
        status = "low"
    elif value > ref["high"]:
        status = "high"

    return {"vital": vital_type, "value": value, "status": status, "reference": ref}

def get_patient_summary(mrn: str) -> dict:
    """Gets a full summary of a patient including vitals, ward, and last update time."""
    if mrn in patient_db:
        return patient_db[mrn]
    return {"error": "Patient not found"}

---
## Cell 5 — OpenAI Tool Definitions
Define the tools in OpenAI's JSON Schema format so the model knows what it can call.

In [11]:
# === CELL 5: OPENAI TOOL DEFINITIONS ===

# TODO: Define the tools list in OpenAI format
# Each tool needs: type="function", function={name, description, parameters}
# parameters uses JSON Schema format with type, properties, required

tools = [
    # TODO: Define lookup_vitals tool
    #   name: "lookup_vitals"
    #   description: "Look up the latest vitals for a patient by MRN"
    #   parameters: mrn (string, required)
    {
        "type": "function",
        "function": {
            "name": "lookup_vitals",
            "description": "Look up the latest vitals for a patient by MRN",
            "parameters": {
                "type": "object",
                "properties": {
                    "mrn": {
                        "type": "string",
                        "description": "The Medical Record Number (MRN) of the patient."
                    }
                },
                "required": ["mrn"]
            }
        }
    },
    # TODO: Define check_abnormal_range tool
    #   name: "check_abnormal_range"
    #   description: "Check if a vital sign value is within normal clinical range"
    #   parameters: vital_type (string, enum of valid types), value (number)
    {
        "type": "function",
        "function": {
            "name": "check_abnormal_range",
            "description": "Check if a vital sign value is within normal clinical range",
            "parameters": {
                "type": "object",
                "properties": {
                    "vital_type": {
                        "type": "string",
                        "enum": ["temperature", "pulse", "spo2", "respiration"]
                    },
                    "value": {
                        "type": "number"
                    }
                },
                "required": ["vital_type", "value"]
            }
        }
    },
    # TODO: Define get_patient_summary tool
    #   name: "get_patient_summary"
    #   description: "Get a full summary of a patient including vitals, ward, and last update time"
    #   parameters: mrn (string, required)
    {
        "type": "function",
        "function": {
            "name": "get_patient_summary",
            "description": "Get a full summary of a patient including vitals, ward, and last update time",
            "parameters": {
                "type": "object",
                "properties": {
                    "mrn": {
                        "type": "string",
                        "description": "The Medical Record Number (MRN) of the patient."
                    }
                },
                "required": ["mrn"]
            }
        }
    }
]

print(f"Defined {len(tools)} tools")

Defined 3 tools


---
## Cell 6 — The Complete Tool Calling Loop
Put it all together: user asks -> model calls tools -> you execute -> model answers.

In [15]:
# === CELL 6: TOOL CALLING LOOP ===

# Map function names to actual functions
available_functions = {
    "lookup_vitals": lookup_vitals,
    "check_abnormal_range": check_abnormal_range,
    "get_patient_summary": get_patient_summary,
}

# Reset messages for tool-calling conversation
messages = [
    {"role": "system", "content": "You are a clinical assistant at KHCC. Help nurses with patient care questions. Use the available tools to look up patient data and check vital sign ranges."}
]

# TODO: Write a function clinical_assistant(user_input) that:
# 1. Adds user message to messages list
# 2. Calls client.chat.completions.create() with model, messages, and tools=tools
# 3. Gets the response message: response.choices[0].message
# 4. Appends the assistant message to messages list
# 5. Checks if response_message.tool_calls is not None
# 6. If yes, for each tool_call:
#    a. Get the function name: tool_call.function.name
#    b. Parse arguments: json.loads(tool_call.function.arguments)
#    c. Call the actual function: available_functions[name](**args)
#    d. Append tool result: {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
# 7. After processing all tool calls, call the API again with updated messages
# 8. Append and return the final assistant response

def clinical_assistant(user_input):
    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    response_message = response.choices[0].message
    messages.append(response_message)

    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            if function_name in available_functions:
                function_to_call = available_functions[function_name]
                function_response = function_to_call(**function_args)
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(function_response),
                    }
                )
            else:
                messages.append(
                    {
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": function_name,
                        "content": "Error: Function not found",
                    }
                )

        # Second API call to get the final answer from the model
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        assistant_response = second_response.choices[0].message.content
        messages.append({"role": "assistant", "content": assistant_response})
        return assistant_response
    else:
        return response_message.content

# TODO: Test with these questions:
# "What are the latest vitals for patient 18887304731609?"
# "Is the patient's temperature normal?"
# "Give me a full summary of patient 21307935080461"
print(clinical_assistant("What are the latest vitals for patient 18887304731609?"))
print("\n")
print(clinical_assistant("Is the patient's temperature normal?"))
print("\n")
print(clinical_assistant("Give me a full summary of patient 21307935080461"))

The latest vital signs for patient 18887304731609 are as follows:

- Temperature: 101.5°F
- Pulse: 95 beats per minute
- Blood Pressure: 130/85 mmHg
- SpO2: 96%
- Respiration Rate: 20 breaths per minute

If you need any more information or assistance, feel free to ask!


The patient's temperature of 101.5°F is considered high and outside the normal range. The typical normal range for body temperature is about 97.8°F to 99.1°F. This indicates that the patient may be experiencing a fever. If you need further assistance or care recommendations, let me know!


Here is the full summary for patient 21307935080461:

- **Ward**: KHCC-3B-ONCOLOGY
- **Last Updated**: January 16, 2026, at 14:00

**Vital Signs**:
- **Temperature**: 98.6°F
- **Pulse**: 72 beats per minute
- **Blood Pressure**: 120/80 mmHg
- **SpO2**: 98%
- **Respiration Rate**: 16 breaths per minute

If you need any additional information or assistance, feel free to ask!


---
## Cell 7 — Multi-Turn with Tools and Memory
Combine memory with tool calling for a full clinical conversation.

In [16]:
# === CELL 7: MULTI-TURN WITH TOOLS AND MEMORY ===

# TODO: Combine memory (Cell 2) with tool calling (Cell 6)
# The clinical_assistant function already maintains messages history!
# Have a conversation:

# Turn 1: "What are the vitals for patient 18887304731609?"
print(clinical_assistant("What are the vitals for patient 18887304731609?"))

# Turn 2: "Is anything abnormal?" (should use check_abnormal_range with context from Turn 1)
print(clinical_assistant("Is anything abnormal?"))

# Turn 3: "Compare with patient 21307935080461"
print(clinical_assistant("Compare with patient 21307935080461"))

# TODO: Uncomment and run the above lines after completing Cell 6

The vital signs for patient 18887304731609 are as follows:

- **Temperature**: 101.5°F
- **Pulse**: 95 beats per minute
- **Blood Pressure**: 130/85 mmHg
- **SpO2**: 96%
- **Respiration Rate**: 20 breaths per minute

If you need further assistance, please let me know!
Here are the findings regarding the vital signs for patient 18887304731609:

- **Temperature**: 101.5°F (high; normal range is 97.8°F to 99.1°F)
- **Pulse**: 95 bpm (normal; normal range is 60 to 100 bpm)
- **Blood Pressure**: 130 (incomplete data, but generally 130/85 is considered elevated but not hypertensive)
- **SpO2**: 96% (normal; normal range is 95% to 100%)
- **Respiration Rate**: 20 breaths/min (normal; normal range is 12 to 20 breaths/min)

The notable abnormality is the elevated temperature indicating a fever. If you have any further questions or need assistance with patient management, please let me know!
Here is a comparison of the vital signs between patients 18887304731609 and 21307935080461:

### Patient 

---
## Stretch Challenge
Add a fourth tool: `recommend_action(vital_type, severity)` that suggests clinical actions based on the vital sign and how far out of range it is.

For example:
- Temperature HIGH + severity "mild" -> "Monitor every 2 hours, administer antipyretic if persistent"
- Temperature HIGH + severity "severe" -> "Immediate blood cultures, notify attending physician"
- SpO2 LOW + severity "mild" -> "Reposition patient, recheck in 15 minutes"
- SpO2 LOW + severity "severe" -> "Initiate supplemental O2, call rapid response team"

In [17]:
# === STRETCH: RECOMMEND_ACTION TOOL ===
# TODO: Implement the recommend_action function and add it to tools list

def recommend_action(vital_type: str, status: str) -> str:
    """Suggests clinical actions based on vital sign type and its status (normal, high, low)."""
    actions = {
        "temperature": {
            "high": "Monitor every 2 hours, administer antipyretic if persistent. If severe, immediate blood cultures and notify attending physician.",
            "low": "Warm patient, monitor temperature frequently. If severe, investigate for hypothermia causes and notify physician.",
            "normal": "Continue routine monitoring."
        },
        "pulse": {
            "high": "Assess for dehydration/pain, review medications. If severe, investigate for arrhythmias and notify physician.",
            "low": "Assess for dizziness/fatigue, review medications. If severe, prepare for possible cardiac intervention and notify physician.",
            "normal": "Continue routine monitoring."
        },
        "spo2": {
            "low": "Reposition patient, encourage deep breathing, provide supplemental O2 per orders. If severe, initiate supplemental O2, call rapid response team.",
            "normal": "Continue routine monitoring."
        },
        "respiration": {
            "high": "Assess for respiratory distress, anxiety, pain. If severe, prepare for respiratory support and notify physician.",
            "low": "Assess for sedation, neurological changes. If severe, stimulate patient, prepare for respiratory support and notify physician.",
            "normal": "Continue routine monitoring."
        }
    }

    if vital_type in actions and status in actions[vital_type]:
        return actions[vital_type][status]
    return "No specific recommendation available for this vital type and status."


# Add the new tool to the existing tools list
tools.append(
    {
        "type": "function",
        "function": {
            "name": "recommend_action",
            "description": "Suggests clinical actions based on a vital sign type and its status (e.g., normal, high, low).",
            "parameters": {
                "type": "object",
                "properties": {
                    "vital_type": {
                        "type": "string",
                        "enum": ["temperature", "pulse", "spo2", "respiration"]
                    },
                    "status": {
                        "type": "string",
                        "enum": ["normal", "high", "low"]
                    }
                },
                "required": ["vital_type", "status"]
            }
        }
    }
)

# Update available_functions map
available_functions["recommend_action"] = recommend_action

print(f"Defined {len(tools)} tools after adding recommend_action")

Defined 4 tools after adding recommend_action


---
### KHCC Connection
This tool calling pattern is the foundation for the clinical AI agents you'll build with LangChain in Session 5. The key concepts:

1. **Conversation Memory** — Maintaining a messages list enables multi-turn dialogue
2. **Tool Definitions** — JSON Schema descriptions let the model know what functions are available
3. **Tool Calling Loop** — The pattern of: call API -> execute tools -> call API again is the core agent loop
4. **Clinical Safety** — Tools provide structured, auditable access to patient data

In Session 5, LangChain will abstract this loop into reusable agent architectures.